# M5 Validation: Time-Windowed Latent Editing
End-to-end M5 validation with A2 callback capture and windowed editing.

In [ ]:
import time
from pathlib import Path
from IPython.display import display
import torch
from helpers.methods.callbacks import StepLatentRecorder, TimeWindowLatentEditor
from helpers.runtime.pipeline import default_torch_dtype_for_device, load_flux_pipeline, resolve_runtime_device
from helpers.runtime.sampling import generate_with_prompt
from helpers.runtime.utility import flush_namespace

In [ ]:
output_dir = Path('../python/examples/callbacks/outputs/m5_time_window_notebook').resolve()
output_dir.mkdir(parents=True, exist_ok=True)

device = resolve_runtime_device()
dtype = default_torch_dtype_for_device(torch.float16, device)
try:
    pipe = load_flux_pipeline(
        torch_dtype=dtype,
        device=device,
        offload_mode="group",
        group_use_stream=True,
        group_low_cpu_mem_usage=False,
        enable_text_encoder_group_offload=True,
        text_encoder_group_offload_type="block_level",
        text_encoder_num_blocks_per_group=1,
        enable_vae_group_offload=True,
        vae_group_offload_type="leaf_level",
        vae_num_blocks_per_group=2,
        enable_vae_slicing=True,
        enable_vae_tiling=True,
        enable_layerwise_casting=True,
        layerwise_storage_dtype="float8_e4m3fn",
        layerwise_compute_dtype=dtype,
        enable_channels_last=True,
        enable_xformers_attention=True,
        enable_torch_compile=False,
    )

    recorder = StepLatentRecorder(store_on_cpu=True, on_hook_error='warn')

    def dampen_edit(latents: torch.Tensor, _step: int, _timestep: float, _kwargs):
        return latents * 0.98

    editor = TimeWindowLatentEditor(
        start_step=1,
        end_step=2,
        edit_fn=dampen_edit,
        on_hook_error='warn',
        clone_before_edit=True,
    )

    def combined_callback(pipe_obj, step_index, timestep, callback_kwargs):
        callback_kwargs = editor.callback(pipe_obj, step_index, timestep, callback_kwargs)
        callback_kwargs = recorder.callback(pipe_obj, step_index, timestep, callback_kwargs)
        return callback_kwargs

    t0 = time.perf_counter()
    with torch.inference_mode():
        out = generate_with_prompt(
            pipe=pipe,
            prompt='a dramatic cliffside lighthouse during golden hour',
            seed=404,
            callback=combined_callback,
            callback_tensor_inputs=['latents'],
            num_inference_steps=3,
            guidance_scale=0.0,
        )
    t1 = time.perf_counter()
    image = out.images[0]
    
    path = output_dir / f'm5_time_window_output_{(t1 - t0):.2f}.png'
    image.save(path)

    display(image)
    print('Saved image:', path)
    print('Captured steps:', recorder.steps())
    print('Edited steps:', editor.applied_steps)
    print('Latent norms:', {step: float(lat.norm().item()) for step, lat in recorder.latents_by_step.items()})
finally:
    flush_namespace(
        globals(),
        "pipe",
        "out",
        "image",
        "recorder",
        "editor",
        verbose=True,
    )